# Aula 1 — RT-PCR: Deteção e Quantificação de Microrganismos em Co-cultura

Na aula anterior analisámos as **curvas de crescimento** de 6 microrganismos em cultura pura e comparámos os seus parâmetros de crescimento.

Nesta aula, utilizamos dados de **RT-PCR (Real-Time PCR)** para verificar:
1. Quais as espécies que **sobreviveram** na co-cultura?
2. Qual foi a sua **abundância relativa** em comparação com a cultura pura?

---

## O que é o RT-PCR?

O **RT-PCR (Real-Time Polymerase Chain Reaction)** é uma técnica molecular que permite **amplificar e quantificar DNA** em tempo real.

O valor **Ct (Cycle Threshold)** representa o número de ciclos de amplificação necessários para que o sinal fluorescente ultrapasse um limiar de deteção:

| Ct | Interpretação |
|---|---|
| Baixo (~10–20) | Muito DNA → espécie muito abundante |
| Alto (~30–35) | Pouco DNA → espécie pouco abundante |
| > 35 | Abaixo do limite de deteção → espécie provavelmente ausente |

---

## Estrutura da Aula

| Exercício | Objetivo |
|---|---|
| **Exercício 1** | Analisar os primers usados — especificidade e qualidade |
| **Exercício 2** | Analisar os resultados de RT-PCR da co-cultura |

---


#### **Questão**

Com base nas previsões que fizeram no final da Aula 0, qual(is) espécie(s) esperam encontrar em maior abundância na co-cultura? E qual(is) acham que pode(m) ter sido eliminada(s)?

# Exercício 1 — Análise dos Primers de RT-PCR

Para que o RT-PCR seja específico, cada espécie deve ser detetada por um **par de primers único** — sequências curtas de DNA que se ligam especificamente ao genoma do organismo alvo.

Neste exercício vamos:
1. Calcular o **reverse complement** de cada primer reverso (necessário para o BLAST)
2. Avaliar a **qualidade** dos primers (comprimento, %GC, temperatura de melting)
3. Identificar **a que espécie** corresponde cada par de primers usando o NCBI BLAST
4. Verificar a **especificidade** dos primers — anelam noutros organismos?


## 0. Importar pacotes necessários

In [1]:
#!pip install biopython

from Bio.Blast import NCBIWWW, NCBIXML #Enviar sequências ao NCBI BLAST e parsear os resultados
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

## 1. Que espécie identifica cada um dos conjuntos de primers de RT-PCR?

### 1.1. Importar dados necessários

In [2]:

# ============================================================
# Pares de primers : (name, target_organism, forward, reverse)
# ============================================================
df_primers = pd.read_csv('primer_pairs.csv', sep=';') # abre e visualiza a lista de primers usados para o RT-PCR
print('Primer List')
display(df_primers)



Primer List


,Forward,Reverse
0,AAAACGGCAAGAAAAAGCAG,ACGCGTGGTTACAGTCTTGCG
1,GTGAAATTATCGCCACGTTCGGGCAA,TCATCGCACCGTCAAAGGAACC
2,CCTTTCTAAGGAAGCGAAGGAT,AATTCTCTTCTCGGTCGCTCTA
3,GAAAGTCCAAGTTTACGCTCAAT,GCTGCACCTAAACTTACACCA
4,GCCTTCTACGTTTCCATCCA,GGCCAAATCGATTCTCAAAA
5,GCTACCACTTCAGAATCATCATC,GCACCTTCAGTCGTAGAGACG


### Porquê calcular o Reverse Complement?

O DNA é uma molécula de dupla cadeia. Os primers são desenhados para anelarem em cadeias opostas:

```
5'─────────────────────────────────3'   (cadeia sentido)
           ←──── Primer Reverse
Primer Forward ────►
3'─────────────────────────────────5'   (cadeia anti-sentido)
```

- O **primer forward** anela na cadeia anti-sentido, lendo na direção 5'→3'
- O **primer reverse** é sintetizado na direção 5'→3' da cadeia anti-sentido

Para fazer **BLAST** e comparar ambos os primers na mesma orientação que o genoma de referência, precisamos de calcular o **reverse complement** do primer reverse.

**Como se calcula?**
1. Substituir cada base pela sua complementar: A↔T, G↔C
2. Inverter a sequência resultante

**Exemplo:**
```
Reverse original:   5'-GCACCTTCAGTCGTAGAGACG-3'
Complementar:       3'-CGTGGAAGTCAGCATCTCTGC-5'
Reverse Complement: 5'-CGTCTCTACGACTGAAGGTGC-3'
```


### 1.1. Gera a sequência reversa complementar de cada primer reverso

In [3]:
def rev_comp(seq):

    """
    Calcula o reverse complement de uma sequência de DNA.
    
    Porquê? Os primers reverse são sintetizados na direção 5'→3' 
    da cadeia complementar. Para fazer BLAST na mesma orientação 
    que o forward, precisamos do reverse complement.
    
    Exemplo: 
        Reverse original:  5'-GCACCTTCAGTCGTAGAGACG-3'
        
        Passo 1 - Complementar:  CGTGGAAGTCAGCATCTCTGC
        Passo 2 - Reverter:      CGTCTCTACGACTGAAGGTGC
        
        Reverse Complement: 5'-CGTCTCTACGACTGAAGGTGC-3'
    """
      
    # Passo 1: Criar dicionário com a conversão complementar
    comp = {'A':'T', 'T':'A', 'G':'C', 'C':'G'} #complementa
    
    # Passo 2: Complementar cada base (A↔T, G↔C)
    complementar = "" # criar uma string nova complementar
    for base in seq: #Iterar cada base
        complementar = complementar + comp[base] #sumar à string complementar a base correspondente no dicionário
    
    print(f"Passo 1 - Complementar: 3'-{complementar}-5'")
    
    # Passo 2: Reverter a sequência (ler de trás para a frente)
    print(reversed(complementar))
    reverso = complementar[::-1] # sequencia[inicio:fim:passo] - vazio assume inicio e fim da sequência
    print(f"Passo 2 - Reverter:     5'-{reverso}-3'")
    
    return reverso # devolver reverse


#chamar a função rev_comp para cada primer reverse, guardando o resultado numa nova coluna do dataframe
for index, row in df_primers.iterrows(): # iterar cada linha de primers
    fwd_seq = row['Forward'] # selecionar a sequência foward
    rev_seq = row['Reverse'] # selecionar a sequência reverse
    
    print(f"\n=== Analisando par: {index} ===")
    print(f"Forward: {fwd_seq}")
    print(f"Reverse: {rev_seq}")
    
    df_primers.loc[index, 'Rev_Comp'] = rev_comp(rev_seq) # criar nova coluna para guardar a sequência criada no mesmo index (usando .loc)

display(df_primers)



=== Analisando par: 0 ===
Forward: AAAACGGCAAGAAAAAGCAG
Reverse: ACGCGTGGTTACAGTCTTGCG
Passo 1 - Complementar: 3'-TGCGCACCAATGTCAGAACGC-5'
Passo 2 - Reverter:     5'-CGCAAGACTGTAACCACGCGT-3'

=== Analisando par: 1 ===
Forward: GTGAAATTATCGCCACGTTCGGGCAA
Reverse: TCATCGCACCGTCAAAGGAACC
Passo 1 - Complementar: 3'-AGTAGCGTGGCAGTTTCCTTGG-5'
Passo 2 - Reverter:     5'-GGTTCCTTTGACGGTGCGATGA-3'

=== Analisando par: 2 ===
Forward: CCTTTCTAAGGAAGCGAAGGAT
Reverse: AATTCTCTTCTCGGTCGCTCTA
Passo 1 - Complementar: 3'-TTAAGAGAAGAGCCAGCGAGAT-5'
Passo 2 - Reverter:     5'-TAGAGCGACCGAGAAGAGAATT-3'

=== Analisando par: 3 ===
Forward: GAAAGTCCAAGTTTACGCTCAAT
Reverse: GCTGCACCTAAACTTACACCA
Passo 1 - Complementar: 3'-CGACGTGGATTTGAATGTGGT-5'
Passo 2 - Reverter:     5'-TGGTGTAAGTTTAGGTGCAGC-3'

=== Analisando par: 4 ===
Forward: GCCTTCTACGTTTCCATCCA
Reverse: GGCCAAATCGATTCTCAAAA
Passo 1 - Complementar: 3'-CCGGTTTAGCTAAGAGTTTT-5'
Passo 2 - Reverter:     5'-TTTTGAGAATCGATTTGGCC-3'

=== Analisando par: 5 ===

,Forward,Reverse,Rev_Comp
0,AAAACGGCAAGAAAAAGCAG,ACGCGTGGTTACAGTCTTGCG,CGCAAGACTGTAACCACGCGT
1,GTGAAATTATCGCCACGTTCGGGCAA,TCATCGCACCGTCAAAGGAACC,GGTTCCTTTGACGGTGCGATGA
2,CCTTTCTAAGGAAGCGAAGGAT,AATTCTCTTCTCGGTCGCTCTA,TAGAGCGACCGAGAAGAGAATT
3,GAAAGTCCAAGTTTACGCTCAAT,GCTGCACCTAAACTTACACCA,TGGTGTAAGTTTAGGTGCAGC
4,GCCTTCTACGTTTCCATCCA,GGCCAAATCGATTCTCAAAA,TTTTGAGAATCGATTTGGCC
5,GCTACCACTTCAGAATCATCATC,GCACCTTCAGTCGTAGAGACG,CGTCTCTACGACTGAAGGTGC


### Parâmetros de qualidade dos primers

Um bom primer para RT-PCR deve respeitar critérios específicos:

| Parâmetro | Valor ideal | Porquê |
|---|---|---|
| **Comprimento** | 18–25 bp | Primers muito curtos são pouco específicos; muito longos têm Tm elevada e podem criar estruturas secundárias |
| **%GC** | 40–60% | Garante estabilidade de ligação adequada |
| **Temperatura de melting (Tm)** | 55–65°C | Par de primers com Tm semelhante permite uma temperatura de annealing consistente |
| **ΔTm entre FWD e REV** | < 5°C | Diferenças grandes prejudicam a eficiência da amplificação |

A **Tm** é calculada pela Regra de Wallace (válida para primers < 20 bp):

$$T_m = 2(A + T) + 4(G + C)$$

**Pergunta:** Algum dos primers apresenta características que possam comprometer a amplificação?


### 1.2. Verificar a qualidade dos primers dentro dos parâmetros standerdizados

In [4]:
def analise_primer(df, index, seq, nome):
    
    """
    Analisa um primer de DNA:
    - Comprimento
    - Contagem de cada base
    - % GC
    - Tm (temperatura de melting) pela regra de Wallace: Tm ≈ 2(A+T) + 4(G+C)
    """
    
    #  ────── Comprimento
    comprimento = len(seq) # contar o número de letras para a sequência
    df.loc[index, 'Comprimento'] = comprimento 
    
    #  ────── Contar cada base usando a função .count()
    A = seq.count('A') 
    T = seq.count('T')
    G = seq.count('G')
    C = seq.count('C')
    
    # ────── % GC (para FW e REV)
    df.loc[index, f'GC %_{nome}'] = (G + C) / comprimento * 100 # soma G + C divide pelo total representa em %
    
    # ────── Tm  (para FW e REV)
    df.loc[index, f'Tm_{nome}'] = 2 * (A + T) + 4 * (G + C) # Aplica a fórmula Tm
     
    return df


# ============================================================
# Analisar todos os primers usando a função analise_primer
# ============================================================
for i, row in df_primers.iterrows():

    df_primers = analise_primer(df_primers, i, row['Forward'], 'FWD')
    df_primers = analise_primer(df_primers, i, row['Reverse'], 'REV')

display(df_primers)

,Forward,Reverse,Rev_Comp,Comprimento,GC %_FWD,Tm_FWD,GC %_REV,Tm_REV
0,AAAACGGCAAGAAAAAGCAG,ACGCGTGGTTACAGTCTTGCG,CGCAAGACTGTAACCACGCGT,21.0,40.000000,56.0,57.142857,66.0
1,GTGAAATTATCGCCACGTTCGGGCAA,TCATCGCACCGTCAAAGGAACC,GGTTCCTTTGACGGTGCGATGA,22.0,50.000000,78.0,54.545455,68.0
2,CCTTTCTAAGGAAGCGAAGGAT,AATTCTCTTCTCGGTCGCTCTA,TAGAGCGACCGAGAAGAGAATT,22.0,45.454545,64.0,45.454545,64.0
3,GAAAGTCCAAGTTTACGCTCAAT,GCTGCACCTAAACTTACACCA,TGGTGTAAGTTTAGGTGCAGC,21.0,39.130435,64.0,47.619048,62.0
4,GCCTTCTACGTTTCCATCCA,GGCCAAATCGATTCTCAAAA,TTTTGAGAATCGATTTGGCC,20.0,50.000000,60.0,40.000000,56.0
5,GCTACCACTTCAGAATCATCATC,GCACCTTCAGTCGTAGAGACG,CGTCTCTACGACTGAAGGTGC,21.0,43.478261,66.0,57.142857,66.0


### Identificação da espécie-alvo por BLAST

O **BLAST (Basic Local Alignment Search Tool)** é uma ferramenta do NCBI que procura sequências similares a uma sequência de entrada nas bases de dados genómicas.

Para um primer de RT-PCR, esperamos encontrar um hit de **100% de identidade** no genoma da espécie-alvo.

**Como interpretar os resultados do BLAST:**

| Campo | Significado |
|---|---|
| **% Identity** | Percentagem de bases iguais entre o primer e o genoma |
| **E-value** | Probabilidade de encontrar um hit por acaso — valores < 0.01 indicam hit significativo |
| **Score** | Pontuação de qualidade do alinhamento |
| **Taxonomy** | Identifica o organismo onde foi encontrado o hit |
| **Gene/Alignments** | Indica em que gene o primer anela |

**Cada grupo irá fazer o BLAST do seu par de primers** e preencher a tabela abaixo.


### 1.2. Verificar a que espécie corresponde cada conjunto de primers (1 conjunto por grupo)

Instruções:
1. Vai a https://blast.ncbi.nlm.nih.gov/ → BLASTn
2. Cola o Forward do teu primer → regista a espécie e gene
    - Verifica na 'Taxonomy' qual a estirpe avaliada
    - No 'Alignments' clica no 'Graphics' para verificar que gene é sequênciada pelos primers
3. Cola o Reverse → confirma que dá hit na mesma espécie
4. Preenche abaixo usando .loc para o teu primer

In [5]:
# ============================================================
# EXERCÍCIO: Preencher após consulta no NCBI BLAST
# ============================================================

# ────── Criar colunas vazias para os resultados
df_primers['Especie_BLAST'] = ""
df_primers['Gene_BLAST'] = ""

# Grupo 1
df_primers.loc[0, 'Especie_BLAST'] = "E_coli_K12"
df_primers.loc[0, 'Gene_BLAST'] = "uidA"
# Grupo 2
df_primers.loc[1, 'Especie_BLAST'] = "S_enterica"
df_primers.loc[1, 'Gene_BLAST'] = "invA"
# Grupo 3
df_primers.loc[2, 'Especie_BLAST'] = "L_acidophilus"
df_primers.loc[2, 'Gene_BLAST'] = "its"
# Grupo 4
df_primers.loc[3, 'Especie_BLAST'] = "C_difficile"
df_primers.loc[3, 'Gene_BLAST'] = "tcdB"
# Grupo 5
df_primers.loc[4, 'Especie_BLAST'] = "S_cerevisiae"
df_primers.loc[4, 'Gene_BLAST'] = "act1"
# Grupo 6
df_primers.loc[5, 'Especie_BLAST'] = "C_albicans"
df_primers.loc[5, 'Gene_BLAST'] = "hwp1"

display(df_primers)

,Forward,Reverse,Rev_Comp,Comprimento,GC %_FWD,Tm_FWD,GC %_REV,Tm_REV,Especie_BLAST,Gene_BLAST
0,AAAACGGCAAGAAAAAGCAG,ACGCGTGGTTACAGTCTTGCG,CGCAAGACTGTAACCACGCGT,21.0,40.000000,56.0,57.142857,66.0,E_coli_K12,uidA
1,GTGAAATTATCGCCACGTTCGGGCAA,TCATCGCACCGTCAAAGGAACC,GGTTCCTTTGACGGTGCGATGA,22.0,50.000000,78.0,54.545455,68.0,S_enterica,invA
2,CCTTTCTAAGGAAGCGAAGGAT,AATTCTCTTCTCGGTCGCTCTA,TAGAGCGACCGAGAAGAGAATT,22.0,45.454545,64.0,45.454545,64.0,L_acidophilus,its
3,GAAAGTCCAAGTTTACGCTCAAT,GCTGCACCTAAACTTACACCA,TGGTGTAAGTTTAGGTGCAGC,21.0,39.130435,64.0,47.619048,62.0,C_difficile,tcdB
4,GCCTTCTACGTTTCCATCCA,GGCCAAATCGATTCTCAAAA,TTTTGAGAATCGATTTGGCC,20.0,50.000000,60.0,40.000000,56.0,S_cerevisiae,act1
5,GCTACCACTTCAGAATCATCATC,GCACCTTCAGTCGTAGAGACG,CGTCTCTACGACTGAAGGTGC,21.0,43.478261,66.0,57.142857,66.0,C_albicans,hwp1


### Automatizar o BLAST com BioPython

Em vez de fazer o BLAST manualmente para cada primer e cada organismo, podemos usar o **BioPython** para enviar os pedidos ao NCBI de forma automatizada.

Os parâmetros usados são otimizados para sequências curtas (primers):

| Parâmetro | Valor | Porquê |
|---|---|---|
| `word_size` | 7 | Mínimo permitido pelo BLASTn — necessário para encontrar alinhamentos em sequências tão curtas |
| `expect` (E-value) | 10 | Valor relaxado para não perder hits em primers curtos |
| `identity_threshold` | 80% | Um primer com < 80% de identidade dificilmente anelará de forma eficiente |

O código vai:
1. Percorrer cada par de primers
2. Para cada organismo, enviar o Forward e o Reverse ao NCBI
3. O NCBI procura essa sequência no genoma do organismo (filtrado pelo taxid)
4. Devolve os melhores alignments encontrados
5. Registar se existe um **HIT** (identidade ≥ 80%) ou **NO HIT**

⚠️ **Nota:** O NCBI pode demorar alguns segundos por pedido. Seja paciente!


## 1.3. Optimizar usando o NCBI

### 1.3.1 Definir parâmetros

In [6]:
# ============================================================
# Organismos em análise: (nome, NCBI taxid)
# ============================================================

ORGANISMS = {
    "E_coli_K12": 83333,
    "S_enterica": 28901,
    "L_acidophilus": 1579,
    "C_difficile": 1496,
    "S_cerevisiae": 4932,
    "C_albicans": 5476
}

# Parâmetros do BLAST
IDENTITY_THRESHOLD = 80.0   # % mínima de identidade para considerar um hit
EVALUE_THRESHOLD   = 10     # valor de E relaxado para sequências curtas
WORD_SIZE          = 7      # "word size" curto para primers

print(f"  {len(df_primers)} pares de primers")
print(f"  {len(ORGANISMS)} organismos")
print(f"  Total de BLASTs: {len(df_primers) * len(ORGANISMS) * 2}") # calcula o total de blasts a realizar (2 por par de primers: forward e reverse complement)

  6 pares de primers
  6 organismos
  Total de BLASTs: 72


### 1.3.2 Aceder ao NCBI

In [ ]:
# ────── Escolher um primer e um organismo para testar
target_name = "S_enterica" # espécie do grupo
row_species = df_primers[df_primers['Especie_BLAST'] == target_name] # filtrar a tabela pela espécie
primer_fw = row_species['Forward']  # sequência Foward
primer_rv = row_species['Reverse']  # sequência Reverse
taxid = ORGANISMS[target_name]       # Tirar o código

print(f"Organismo: {target_name} (taxid: {taxid})")

# ────── Passo 1: Enviar o primer ao NCBI
for primer_seq in [primer_fw, primer_rv]:
    result_handle = NCBIWWW.qblast(
        program="blastn",
        database="nt",
        sequence=primer_seq,
        entrez_query=f"txid{taxid}[ORGN]",
        word_size=WORD_SIZE,
        expect=EVALUE_THRESHOLD,
        megablast=False,
        hitlist_size=5,
    )

    # ────── 1. Parsear o resultado (transformar o XML em objetos Python)
    blast_record = NCBIXML.read(result_handle)

    # ────── Variável para armazenar apenas o objeto do alinhamento
    hit_flag = None

    for alignment in blast_record.alignments: # retirar alinhamento
        if alignment.hsps: # evitar erros caso não haja HSPs
            hsp = alignment.hsps[0] # Verificamos o primeiro HSP - High Scoring Pairs- para validar a identidade (geralmente o melhor).
            identity_pct = (hsp.identities / len(primer_seq)) * 100 # confirmar a identidade (todos os nucleótidos alinham)
            
            if identity_pct >= IDENTITY_THRESHOLD:
                hit_flag = 'HIT'
                break # Pára o loop assim que encontra o primeiro

    if hit_flag:
        print(f"{hit_flag}: {alignment.title}")
    else:
        print(f"NO HIT")

    time.sleep(0.5)  # pausa para respeitar o NCBI

Organismo: S_enterica (taxid: 28901)


## 2 — Especificidade dos Primers

Um primer específico é aquele que **anela apenas** no organismo-alvo e não noutras espécies presentes na amostra.

Num experimento de co-cultura com **6 espécies diferentes**, a especificidade é crítica — um primer que anele em múltiplas espécies daria resultados falsos positivos.

**O que vamos testar:**
- Para cada par de primers, vamos correr o BLAST contra os genomas de **todas as 6 espécies**
- Se o Forward **e** o Reverse anelam na mesma espécie → potencial amplificação (cross-reactivity)
- Se apenas um anela → normalmente não há amplificação

**Pergunta:** Com base nos resultados esperados, seria possível usar estes primers numa amostra que contivesse **bactérias do solo**? E numa amostra **humana**? Porquê?


# Primer Specificity Checker via NCBI BLAST

Testa cada par de primers contra 6 organismos-alvo para verificar cross-reactivity.

**Organismos:**
- *E. coli* K12
- *Salmonella enterica*
- *Lactobacillus acidophilus*
- *Clostridioides difficile*
- *Saccharomyces cerevisiae*
- *Candida albicans*

**Lógica:** Para cada par, faz BLASTn do Forward e do RevComp do Reverse contra cada organismo. Se **ambos** anelam → potencial amplificação (cross-react se não for o alvo).


In [ ]:
print('O grupo vai testar:')

print(f"Primers da espécie: {target_name}")
print(f"Forward: {primer_fw}")
print(f"Reverse: {primer_rv}")
print(f"\nVou testar contra {len(ORGANISMS)} organismos...\n")

In [ ]:
# ============================================================
# Correr o BLAST contra TODOS os organismos
# Mesmo código que usámos antes, agora dentro de um loop
# ============================================================

for org_name, taxid in ORGANISMS.items(): # loop para organismos (tira os items do dicionário)
    
    print(f"\n=== Testar contra {org_name} (taxid: {taxid}) ===")

    for primer_label, primer_seq in [("Fwd", primer_fw), ("Rev", primer_rv)]: # cria listas para a label e seq de primer a iterar. Primeiro Foward e depois reverse.
        
        print(f"Primer {primer_label}: {primer_seq}")
        
        # Enviar ao NCBI
        result_handle = NCBIWWW.qblast(
            program="blastn",
            database="nt",
            sequence=primer_seq,
            entrez_query=f"txid{taxid}[ORGN]",
            word_size=WORD_SIZE,
            expect=EVALUE_THRESHOLD,
            megablast=False,
            hitlist_size=5,
        )
        
        # Parsear o resultado
        blast_record = NCBIXML.read(result_handle)
        
        # Procurar hits válidos
        hit_flag = None
        
        for alignment in blast_record.alignments:
            if alignment.hsps:
                hsp = alignment.hsps[0]
                identity_pct = (hsp.identities / len(primer_seq)) * 100
                
                if identity_pct >= IDENTITY_THRESHOLD:
                    print(f"  → HIT! ({identity_pct:.1f}%) — {alignment.title[:80]}")
                    hit_flag = 'HIT'
                    break
        
        if hit_flag:
            print(f"{hit_flag}: {alignment.title}")
        else:
            print(f"NO HIT")
        
        time.sleep(0.5)  # pausa para respeitar o NCBI

print("\n\nBLAST concluído!")

# Exercício 2 — Análise dos Resultados de RT-PCR

Após verificar a qualidade e especificidade dos primers, aplicamos a técnica de RT-PCR para **quantificar a abundância relativa** de cada espécie na co-cultura.

---

## Desenho Experimental

| Condição | Descrição | Papel |
|---|---|---|
| **Cultura pura** | Cada espécie crescida isoladamente | Controlo (valor de referência) |
| **Co-cultura** | As 6 espécies crescidas juntas | Amostra a analisar |

Para cada amostra, medimos dois valores de Ct:
- **Ct_gene**: Ct do primer específico para cada espécie (diz-nos quanto DNA dessa espécie existe)
- **Ct_ref**: Ct do gene de referência (16S rRNA para bactérias, 18S rRNA para leveduras) — normaliza pela quantidade total de DNA extraído

---

## Pipeline de Análise

```
Ct_gene + Ct_ref
       │
       ▼
   ΔCt = Ct_gene − Ct_ref          (normalização)
       │
       ▼
   ΔΔCt = ΔCt(co-cultura) − ΔCt(cultura pura)   (comparação)
       │
       ▼
   Fold Change = 2^(−ΔΔCt)         (quantificação)
       │
       ▼
   Testes estatísticos (t-test, ANOVA)
```


## 1. Importar dados

In [ ]:
df_coculture = pd.read_csv('rtpcr_cocultura.csv')
print(df_coculture)

# ────── Verifica quais as condições a analisar
print(f"\nCondições: {df_coculture['Condicao'].unique()}") # utiliza a função unique
# ────── Verifica se temos dados para todas as estirpes
print(f"Primers:   {df_coculture['Primer'].unique()}") # informação na coluna primer


## 2. Visualizar os Ct brutos

Comparar o Ct de cada espécie entre cultura pura e co-cultura.

- Ct **sobe** na co-cultura → espécie está **menos abundante** (menos DNA)
- Ct **desce** na co-cultura → espécie está **mais abundante** (mais DNA)
- Ct **> 35** → espécie **não detetada** (eliminada?)

In [ ]:
#  ────── Calcular média e SD por condição e primer ────── 

ct_stats = df_coculture.groupby(['Primer', 'Condicao'])['Ct_gene'].agg(['mean', 'std']).round(2) 
    # Agrupar por Primer e Condição - groupby
    # Selecionar a coluna de interesse
    # Agregar calculando média e desvio padrão - agg
print(ct_stats)

# ─── Melhorar a tabela 
ct_stats.columns = ['Ct_mean', 'Ct_std']
ct_stats = ct_stats.reset_index()

print("Ct médio por espécie e condição:\n")
print(ct_stats)



### Interpretação visual dos Ct

No gráfico seguinte, comparamos os Ct da cultura pura (azul) e da co-cultura (coral) para cada espécie.

**O que observar:**
- Barras **iguais** → a espécie está na mesma abundância nas duas condições
- Barra da co-cultura **mais alta** → espécie **menos abundante** (Ct subiu = menos DNA)
- Barra da co-cultura **mais baixa** → espécie **mais abundante** (Ct desceu = mais DNA)
- Barra da co-cultura **acima de 35** (linha vermelha) → espécie provavelmente **não detetada**

**Pergunta:** Com base no gráfico, qual(is) espécie(s) parecem ter sido eliminadas? E qual(is) parecem ter beneficiado da co-cultura?


In [ ]:
# ─── Gráfico: Ct pura vs co-cultura lado a lado
fig, ax = plt.subplots(figsize=(10, 6))

# ─── Definir espécies
primers = df_coculture['Primer'].unique() # através da coluna 'Primer' usando unique()
x = np.arange(len(primers)) # valores de X com base no número de amostras
width = 0.35 # largura das colunas do gráfico

# ───Comparar dados de cultura pura com crescimento em co-cultura
pura = ct_stats[ct_stats['Condicao'] == 'Cultura_pura'] # Filtrar os dados para cultura pura
co = ct_stats[ct_stats['Condicao'] == 'Co-cultura']# Filtrar os dados para co-cultura

# ─── Usar ax.bar para criar plot
ax.bar(x - width/2, pura['Ct_mean'].values, width, yerr=pura['Ct_std'].values, # X = número do primer em análise - a metade da largura,  Y - values de Ct_mean, yerr - values de Ct_std
       capsize=5, label='Cultura pura', color='steelblue', edgecolor='black')

ax.bar(x + width/2, co['Ct_mean'].values, width, yerr=co['Ct_std'].values, # X = número do primer em análise + a metade da largura
       capsize=5, label='Co-cultura', color='coral', edgecolor='black')

# ─── Linha de corte para "não detetado"
ax.axhline(y=35, color='red', linestyle='--', alpha=0.5, label='Limite de deteção (Ct=35)')

ax.set_xticks(x)
ax.set_xticklabels(primers)
ax.set_ylabel('Ct (ciclos)')
ax.set_title('Ct do gene-alvo: Cultura pura vs Co-cultura')
ax.legend()

plt.tight_layout()
plt.show()


## 3. Calcular ΔCt

**ΔCt = Ct_gene − Ct_ref**

O Ct_ref (16S/18S rRNA) mede a quantidade total de DNA na amostra.
Subtrair o Ct_ref normaliza as diferenças na quantidade de DNA extraído.

- ΔCt **baixo** → espécie abundante relativamente ao DNA total
- ΔCt **alto** → espécie pouco abundante

In [ ]:
# Para calcular o ΔCt precisamos de retirar os valores de Ct_ref da database 'df_coculture'
#print(df_coculture)

# ─── Calcular ΔCt para cada medição
df_coculture['delta_ct'] = df_coculture['Ct_gene'] - df_coculture['Ct_ref'] # calcular criando uma nova coluna

print("ΔCt = Ct_gene − Ct_ref\n")
print(df_coculture[['Condicao', 'Primer', 'Replicate', 'Ct_gene', 'Ct_ref', 'delta_ct']])


## 3. Calcular ΔΔCt

**ΔΔCt = ΔCt(co-cultura) − ΔCt(cultura pura)**

Compara cada espécie na co-cultura com a sua cultura pura (controlo).

- ΔΔCt **= 0** → mesma abundância nas duas condições
- ΔΔCt **> 0** → menos abundante na co-cultura (Ct subiu)
- ΔΔCt **< 0** → mais abundante na co-cultura (Ct desceu)


In [ ]:
# Passo 1: Calcular ΔCt médio da cultura pura (controlo) para cada espécie
delta_ct_pura = df_coculture[df_coculture['Condicao'] == 'Cultura_pura'].groupby('Primer')['delta_ct'].mean()

print("ΔCt médio da cultura pura (controlo):\n")
print(delta_ct_pura.round(2))


### Passo 2: Calcular ΔΔCt para cada réplica da co-cultura

Usando o ΔCt médio da cultura pura como referência, calculamos o **ΔΔCt** para cada réplica da co-cultura:

$$\Delta\Delta Ct = \Delta Ct_{(co\text{-}cultura)} - \overline{\Delta Ct}_{(cultura\ pura)}$$

**Interpretação:**
- ΔΔCt = 0 → abundância inalterada
- ΔΔCt > 0 → espécie **menos** abundante na co-cultura
- ΔΔCt < 0 → espécie **mais** abundante na co-cultura


In [ ]:
# Passo 2: Calcular ΔΔCt para cada medição da co-cultura
# ΔΔCt = ΔCt(co-cultura) − ΔCt médio(cultura pura)

df_co = df_coculture.loc[df_coculture['Condicao'] == 'Co-cultura'].copy()

# Para cada linha, subtrair o ΔCt médio da cultura pura da mesma espécie
df_co['delta_ct_pura'] = df_co['Primer'].map(delta_ct_pura)
df_co['delta_delta_ct'] = df_co['delta_ct'] - df_co['delta_ct_pura']

print("ΔΔCt = ΔCt(co-cultura) − ΔCt(cultura pura)\n")
df_co[['Primer', 'Replicate', 'delta_ct', 'delta_ct_pura', 'delta_delta_ct']].round(2)


## 5. Calcular Fold Change

**FC = 2^(−ΔΔCt)**

O fold change indica quanto a abundância mudou na co-cultura em relação à cultura pura.

- FC **= 1** → sem alteração
- FC **> 1** → espécie mais abundante na co-cultura
- FC **< 1** → espécie menos abundante na co-cultura
- FC **≈ 0** → espécie eliminada/não detetada

In [ ]:
# Calcular Fold Change
df_co['fold_change'] = 2 ** (-df_co['delta_delta_ct'])

print("Fold Change = 2^(−ΔΔCt)\n")
df_co[['Primer', 'Replicate', 'delta_delta_ct', 'fold_change']].round(3)


In [ ]:
# Resumo: média e SD do fold change por espécie
fc_stats = df_co.groupby('Primer')['fold_change'].agg(['mean', 'std']).round(3)
fc_stats.columns = ['FC_mean', 'FC_std']
fc_stats = fc_stats.sort_values('FC_mean', ascending=False)

print("Fold Change por espécie (co-cultura vs cultura pura):\n")
print(fc_stats)


### Visualização do Fold Change

O gráfico seguinte representa visualmente o **fold change** de cada espécie, com código de cores:

| Cor | Significado | FC |
|---|---|---|
| 🟢 Verde | Espécie **aumentou** na co-cultura | FC > 1.5 |
| 🔵 Azul | **Sem grande alteração** | FC ≈ 0.5–1.5 |
| 🟠 Salmon | Espécie **reduziu** na co-cultura | FC < 0.5 |
| 🔴 Vermelho | Espécie **eliminada** / não detetada | FC ≈ 0 |

A linha tracejada indica FC = 1 (sem alteração).


In [ ]:
# Gráfico do Fold Change
fig, ax = plt.subplots(figsize=(10, 6))

especies = fc_stats.index.tolist()
means = fc_stats['FC_mean'].values
stds = fc_stats['FC_std'].values

colors = []
for m in means:
    if m > 1.5:
        colors.append('forestgreen')   # aumentou
    elif m < 0.1:
        colors.append('red')           # eliminado
    elif m < 0.5:
        colors.append('salmon')        # reduzido
    else:
        colors.append('steelblue')     # sem grande alteração

ax.bar(especies, means, yerr=stds, capsize=5, color=colors, edgecolor='black')
ax.axhline(y=1, color='gray', linestyle='--', label='Sem alteração (FC=1)')
ax.set_ylabel('Fold Change (2^−ΔΔCt)')
ax.set_title('Abundância relativa na co-cultura vs cultura pura')
ax.set_xticklabels(especies, rotation=45, ha='right')
ax.legend()

plt.tight_layout()
plt.show()

print("Verde  = espécie AUMENTOU na co-cultura")
print("Azul   = sem grande alteração")
print("Salmon = espécie REDUZIU na co-cultura")
print("Red    = espécie ELIMINADA / não detetada")


## 6. Teste t — Cultura pura vs Co-cultura

Para cada espécie, comparar o ΔCt entre as duas condições.

- H₀: ΔCt(pura) = ΔCt(co-cultura) → não houve alteração
- Se p < 0.05 → a diferença é significativa


In [ ]:
# t-test para cada espécie: cultura pura vs co-cultura
resultados_ttest = []

for primer in df_coculture['Primer'].unique():
    
    dct_pura = df_coculture.loc[(df_coculture['Primer'] == primer) & (df_coculture['Condicao'] == 'Cultura_pura'), 'delta_ct'].values
    dct_co   = df_coculture.loc[(df_coculture['Primer'] == primer) & (df_coculture['Condicao'] == 'Co-cultura'), 'delta_ct'].values
    
    t_stat, p_value = stats.ttest_ind(dct_pura, dct_co)
    
    fc_mean = df_co.loc[df_co['Primer'] == primer, 'fold_change'].mean()
    
    resultados_ttest.append({
        'Especie': primer,
        'ΔCt_pura': round(dct_pura.mean(), 2),
        'ΔCt_co': round(dct_co.mean(), 2),
        'FC': round(fc_mean, 3),
        't_stat': round(t_stat, 3),
        'p_value': round(p_value, 6),
        'Significativo': '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else 'ns'
    })

df_ttest = pd.DataFrame(resultados_ttest)

print("t-test: Cultura pura vs Co-cultura (por espécie)\n")
print(df_ttest.to_string(index=False))

print("\n*** p<0.001  |  ** p<0.01  |  * p<0.05  |  ns = não significativo")


## 7. ANOVA One-way

Compara o fold change entre **todas as espécies** simultaneamente.

Pergunta: "As espécies respondem de forma diferente à co-cultura?"

- H₀: todas as espécies têm o mesmo fold change
- Se p < 0.05 → pelo menos uma espécie responde de forma diferente


In [ ]:
# Separar fold change por espécie
grupos = []
nomes = []

for primer in df_co['Primer'].unique():
    fc_valores = df_co.loc[df_co['Primer'] == primer, 'fold_change'].values
    grupos.append(fc_valores)
    nomes.append(primer)

# ANOVA One-way
f_stat, p_value = stats.f_oneway(*grupos)

print("ANOVA One-way (Fold Change entre espécies):")
print(f"  F = {f_stat:.4f}")
print(f"  p = {p_value:.8f}")

if p_value < 0.05:
    print("\n→ p < 0.05: As espécies respondem de forma DIFERENTE à co-cultura!")
else:
    print("\n→ p ≥ 0.05: Sem diferenças significativas entre espécies")


# Conclusões

Respondam em grupo às seguintes questões:

1. **Sobrevivência:** Todas as 6 espécies sobreviveram à co-cultura? Alguma foi eliminada?

2. **Vencedores e perdedores:** Qual a espécie com maior FC? Qual a que teve menor FC? O que pode explicar estes resultados biologicamente?

3. **Comparação com as previsões da Aula 0:** As espécies com maior taxa de crescimento (μ_max) na Aula 0 foram as que dominaram na co-cultura? Existe correlação entre os parâmetros de crescimento e a sobrevivência em co-cultura?

4. **Significado estatístico:** Para quais espécies a diferença entre cultura pura e co-cultura é estatisticamente significativa (p < 0.05)? E para quais não é?

5. **Mecanismos ecológicos:** Com base nos resultados:
   - Existe evidência de **competição** entre espécies?
   - Existe alguma espécie que possa estar a **favorecer** outra (mutualismo)?
   - A patogenicidade influenciou a sobrevivência?

---

**Na próxima aula:** Vamos aprofundar a análise das interações entre espécies e discutir como os resultados se relacionam com a ecologia do microbioma humano.
